### NB_OTT_Bronze

#### Objetivo:
##### Ingestar el dataset batch de tickets OTT desde OneLake y persistirlo en la capa Bronze del Lakehouse.

#### Entrada:
#####     Files/Bronze/Raw/Tickets/synthetic_ott_tickets.csv
#### Salida:
#####     Bronze.Tickets
#####
#### Este notebook forma parte del pipeline operacional.
#####
##### Principios aplicados:
##### - Schema explícito para evitar inferencias variables.
##### - Conservación de los datos próximos al formato original.
##### - Validaciones automáticas de estructura y volumen.
##### - Sin lógica de limpieza de negocio: esa responsabilidad pertenece a NB_OTT_BronzeToSilver.

In [ ]:
from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType
)

In [ ]:
# ============================================================
# 1. Configuración de la fuente
# ============================================================

INPUT_PATH = (
    "Files/Bronze/Raw/Tickets/"
    "synthetic_ott_tickets.csv"
)

TARGET_TABLE = "Bronze.Tickets"

print("Bronze ingestion configuration")
print("------------------------------")
print("Input:", INPUT_PATH)
print("Target:", TARGET_TABLE)

In [ ]:
# ============================================================
# 2. Definición explícita del schema
#
# Se evita inferSchema=True para garantizar que el mismo fichero siempre se interprete de manera consistente.
#
# created_at y resolved_at se mantienen como StringType porque Bronze conserva los datos próximos al formato original. La conversión a timestamp se realiza en Silver.
# ============================================================

ticket_schema = StructType([
    StructField(
        "ticket_id",
        StringType(),
        True
    ),
    StructField(
        "created_at",
        StringType(),
        True
    ),
    StructField(
        "title",
        StringType(),
        True
    ),
    StructField(
        "description",
        StringType(),
        True
    ),
    StructField(
        "priority",
        StringType(),
        True
    ),
    StructField(
        "status",
        StringType(),
        True
    ),
    StructField(
        "category",
        StringType(),
        True
    ),
    StructField(
        "platform",
        StringType(),
        True
    ),
    StructField(
        "device_type",
        StringType(),
        True
    ),
    StructField(
        "country",
        StringType(),
        True
    ),
    StructField(
        "service",
        StringType(),
        True
    ),
    StructField(
        "assigned_team",
        StringType(),
        True
    ),
    StructField(
        "error_code",
        StringType(),
        True
    ),
    StructField(
        "resolved_at",
        StringType(),
        True
    ),
    StructField(
        "incident_id",
        StringType(),
        True
    ),
    StructField(
        "is_isolated",
        BooleanType(),
        True
    ),
    StructField(
        "is_duplicate",
        BooleanType(),
        True
    )
])

In [ ]:
# ============================================================
# 3. Lectura del fichero raw
#
# En la capa Bronze realizamos la ingesta batch a partir del fichero csv con los tickets de los incidentes, sin limpieza de datos.
# ============================================================

df_bronze = (
    spark.read
    .option(
        "header",
        True
    )
    .schema(
        ticket_schema
    )
    .csv(
        INPUT_PATH
    )
)

input_count = df_bronze.count()

print(
    f"Records read from source: {input_count}"
)

assert input_count > 0, (
    "The source file contains no records."
)

In [ ]:
# ============================================================
# 4. Validación de la estructura
#
# Comprobamos que todas las columnas esperadas estén disponibles antes de guardar capa Bronze.
# ============================================================

expected_columns = [
    "ticket_id",
    "created_at",
    "title",
    "description",
    "priority",
    "status",
    "category",
    "platform",
    "device_type",
    "country",
    "service",
    "assigned_team",
    "error_code",
    "resolved_at",
    "incident_id",
    "is_isolated",
    "is_duplicate"
]

actual_columns = df_bronze.columns

missing_columns = [
    column
    for column in expected_columns
    if column not in actual_columns
]

unexpected_columns = [
    column
    for column in actual_columns
    if column not in expected_columns
]

assert len(missing_columns) == 0, (
    f"Missing expected columns: {missing_columns}"
)

assert len(unexpected_columns) == 0, (
    f"Unexpected columns found: {unexpected_columns}"
)


print("Schema validation")
print("-----------------")
print(
    "Expected columns:",
    len(expected_columns)
)
print(
    "Actual columns:",
    len(actual_columns)
)
print(
    "Missing columns:",
    len(missing_columns)
)
print(
    "Unexpected columns:",
    len(unexpected_columns)
)

In [ ]:
# ============================================================
# 5. Validaciones básicas de calidad de entrada
#
# Bronze no elimina estos registros.
#
# Los indicadores se calculan para:
# - observabilidad
# - trazabilidad
# - validación posterior en Silver.
# ============================================================

null_ticket_ids = (
    df_bronze
    .filter(
        F.col("ticket_id").isNull()
    )
    .count()
)

null_descriptions = (
    df_bronze
    .filter(
        F.col("description").isNull()
    )
    .count()
)

duplicate_ticket_ids = (
    df_bronze
    .groupBy(
        "ticket_id"
    )
    .count()
    .filter(
        F.col("ticket_id").isNotNull()
        &
        (
            F.col("count") > 1
        )
    )
    .count()
)

print("Bronze input quality")
print("--------------------")
print(
    "Records:",
    input_count
)
print(
    "Null ticket IDs:",
    null_ticket_ids
)
print(
    "Null descriptions:",
    null_descriptions
)
print(
    "Duplicated ticket IDs:",
    duplicate_ticket_ids
)

In [ ]:
# ============================================================
# 6. Metadatos técnicos de ingesta
#
# Se añaden únicamente metadatos técnicos (_ingested_at y _source_file). Los campos originales del dataset se conservan sin transformaciones.
# _source_file: contiene el nombre del fichero con los datos de entrada en batch
# _ingested_at: contiene la fecha en la que se realizó la ingesta de los datos
# ============================================================

df_bronze = (
    df_bronze
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.lit(
            "synthetic_ott_tickets.csv"
        )
    )
)

In [ ]:
# ============================================================
# 7. Persistencia en Bronze
#
# Para el TFM, Bronze se reconstruye mediante overwrite ya que esto permite que las ejecuciones del pipeline sean reproducibles.
# En producción podría evolucionar a una estrategia incremental o append.
# ============================================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS Bronze"
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"{TARGET_TABLE} written successfully."
)

In [ ]:
# ============================================================
# 8. Validación posterior a escritura
#
# Verificamos que no se hayan perdido registros entre la lectura raw y la persistencia Bronze.
# ============================================================

df_saved = spark.table(
    TARGET_TABLE
)

saved_count = (
    df_saved.count()
)

assert saved_count == input_count, (
    f"Expected {input_count} records "
    f"but persisted {saved_count}."
)

saved_columns = set(
    df_saved.columns
)

for required_column in expected_columns:

    assert required_column in saved_columns, (
        f"Column {required_column} missing "
        f"from {TARGET_TABLE}."
    )


print("Bronze ingestion completed")
print("--------------------------")
print(
    "Source records:",
    input_count
)
print(
    "Persisted records:",
    saved_count
)
print(
    "Null ticket IDs:",
    null_ticket_ids
)
print(
    "Duplicate ticket IDs:",
    duplicate_ticket_ids
)

print(
    "\nNB_OTT_Bronze "
    "completed successfully."
)